# 05b — SARIMA Order Selection for GHI Forecasting

This notebook documents the statistical justification for the SARIMA order
used in `05_sarima_baseline.py`.

The model is fit directly on the **raw hourly GHI series** (W/m²) — the same
tabular ground-truth (`ground_10min_utc_{site}.parquet`, resampled to hourly)
used to train and evaluate every other model in this project. No clear-sky
normalisation or derived index is used: SARIMA forecasts the same physical
quantity (GHI) that persistence and the deep-learning models forecast.

**Pipeline:**
1. Load raw hourly GHI on the training period.
2. Stationarity tests (ADF, KPSS) → determine `d` and `D`.
3. ACF/PACF plots on `ΔGHI` and `Δ_24 ΔGHI` → initial guess for `p,q,P,Q`.
4. AIC/BIC grid search over candidate orders.
5. Residual diagnostics on the chosen model.
6. Summary and final order decision.


In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Suppress convergence warnings during grid search
warnings.filterwarnings("ignore")

SITE = "elpaso"   # change to "uniandes" to repeat analysis
GROUND_DIR   = PROJECT_ROOT / "data" / "ground_aligned"
MANIFEST_DIR = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"

print(f"Site: {SITE}")


## 1. Load hourly GHI series (raw, same tabular ground-truth as the DL models)


In [ ]:
ground = pd.read_parquet(GROUND_DIR / f"ground_10min_utc_{SITE}.parquet")
ghi_hourly = ground["ghi"].resample("1h").mean().dropna()

# Use training period only (matches the manifest train split for h1)
man_dir = MANIFEST_DIR / SITE / "h1"
train_man = pd.read_parquet(man_dir / "manifest_train.parquet")
t_labels  = pd.to_datetime(train_man["t_label"], utc=True)
train_start, train_end = t_labels.min(), t_labels.max()

ghi_train = ghi_hourly.loc[(ghi_hourly.index >= train_start) & (ghi_hourly.index <= train_end)]
print(f"Training period: {train_start.date()} → {train_end.date()}")
print(f"ghi_train: {len(ghi_train):,} hourly observations  |  "
      f"mean={ghi_train.mean():.1f}  std={ghi_train.std():.1f} W/m²")


In [ ]:
# Quick visual of the raw hourly GHI series (one representative window)
sample = ghi_train["2023-06-01":"2023-06-14"]
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(sample.index, sample.values, lw=0.8)
ax.set_ylabel("GHI (W/m²)")
ax.set_title(f"{SITE} — raw hourly GHI (two weeks, training set)")
fig.tight_layout()
plt.show()


## 2. Stationarity tests

SARIMA requires choosing the integration orders `d` (regular) and `D` (seasonal).

* **ADF test** (H0: unit root, i.e. non-stationary) — reject H0 → stationary.
* **KPSS test** (H0: stationary) — fail to reject H0 → stationary.

We test on raw `GHI`, `ΔGHI`, and `Δ_24 GHI` (one seasonal lag difference).
Note that raw GHI is bounded at zero and has a strong, large-amplitude diurnal
cycle (it is exactly zero at night) — its stationarity properties differ
substantially from a normalised clear-sky index, so the integration and
AR/MA orders must be re-derived from this series rather than reused.


In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def stationarity_report(series: pd.Series, label: str) -> None:
    s = series.dropna()
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(s, autolag="AIC")
    kpss_stat, kpss_p, _, kpss_crit = kpss(s, regression="c", nlags="auto")

    adf_conc  = "STATIONARY" if adf_p  < 0.05 else "non-stationary"
    kpss_conc = "STATIONARY" if kpss_p > 0.05 else "non-stationary"
    print(f"{label}")
    print(f"  ADF:  stat={adf_stat:.4f}  p={adf_p:.4f}  → {adf_conc}")
    print(f"  KPSS: stat={kpss_stat:.4f}  p={kpss_p:.4f}  → {kpss_conc}")
    print()

stationarity_report(ghi_train,                     "GHI (raw)")
stationarity_report(ghi_train.diff().dropna(),     "ΔGHI  (d=1)")
stationarity_report(ghi_train.diff(24).dropna(),   "Δ_24 GHI  (D=1)")
stationarity_report(
    ghi_train.diff(24).diff().dropna(),             "Δ Δ_24 GHI  (d=1, D=1)"
)


**Interpretation:** if `Δ Δ_24 GHI` is stationary by both tests, `d=1, D=1` is appropriate.
Over-differencing (both tests showing stationary for `GHI` itself) would suggest `d=0`.
Re-evaluate the chosen `(d, D)` against the test results above — do not assume the
clear-sky-index orders transfer to raw GHI.


## 3. ACF / PACF plots

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

d_ghi = ghi_train.diff(24).diff().dropna()   # Δ Δ_24 GHI

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
plot_acf(d_ghi, lags=50, ax=ax1, title="ACF — Δ Δ₂₄ GHI (lags 0–50)")

ax2 = fig.add_subplot(gs[0, 1])
plot_pacf(d_ghi, lags=50, ax=ax2, title="PACF — Δ Δ₂₄ GHI (lags 0–50)", method="ywm")

# Zoom in on seasonal lags (multiples of 24)
ax3 = fig.add_subplot(gs[1, 0])
plot_acf(d_ghi, lags=72, ax=ax3, title="ACF — Δ Δ₂₄ GHI (lags 0–72, seasonal visible)")
ax3.axvline(24, color="r", ls="--", lw=0.8, label="lag=24")
ax3.axvline(48, color="r", ls="--", lw=0.8)
ax3.legend(fontsize=8)

ax4 = fig.add_subplot(gs[1, 1])
plot_pacf(d_ghi, lags=72, ax=ax4, title="PACF — Δ Δ₂₄ GHI (lags 0–72)", method="ywm")
ax4.axvline(24, color="r", ls="--", lw=0.8)
ax4.axvline(48, color="r", ls="--", lw=0.8)

fig.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / f"sarima_acf_pacf_{SITE}.png", dpi=120)
plt.show()
print("Saved ACF/PACF figure to results/")


**Reading guide:**
- Non-seasonal part (low lags):  
  PACF cuts off at lag `p` → AR(p). ACF cuts off at lag `q` → MA(q).
- Seasonal part (lags 24, 48, ...):  
  PACF cuts off at seasonal lag `P` × 24. ACF cuts off at seasonal lag `Q` × 24.

## 4. AIC/BIC grid search

Candidate non-seasonal orders: p ∈ {0,1,2,3}, q ∈ {0,1,2,3}  
Candidate seasonal orders:     P ∈ {0,1},    Q ∈ {0,1}  
Fixed: d=1, D=1, m=24.

> **Runtime note**: each model fit on 2 years of hourly data (~17k obs) takes ~20–60 s.
> The full grid (≈64 combinations) may take 30–60 min. Narrow the ranges if needed.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import itertools
import time

# Narrow the search to keep runtime reasonable — expand if needed
P_RANGE = [0, 1, 2]
Q_RANGE = [0, 1, 2]
p_RANGE = [0, 1, 2, 3]
q_RANGE = [0, 1, 2, 3]

D, d = 1, 1
m    = 24

# Sub-sample to speed up search: use 18 months of data
ghi_search = ghi_train.iloc[-18*30*24:]   # approx last 18 months
print(f"Grid search on {len(ghi_search):,} obs (last 18 months of training)")
print(f"Grid size: {len(p_RANGE)*len(q_RANGE)*len(P_RANGE)*len(Q_RANGE)} models\n")

results = []
total = len(p_RANGE) * len(q_RANGE) * len(P_RANGE) * len(Q_RANGE)
done  = 0

for p, q, P, Q in itertools.product(p_RANGE, q_RANGE, P_RANGE, Q_RANGE):
    if p == 0 and q == 0:   # degenerate model
        continue
    t0 = time.time()
    try:
        mod = SARIMAX(
            ghi_search,
            order=(p, d, q),
            seasonal_order=(P, D, Q, m),
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fit = mod.fit(disp=False, maxiter=150)
        results.append({
            "order": (p, d, q),
            "seasonal": (P, D, Q, m),
            "aic": fit.aic,
            "bic": fit.bic,
            "converged": fit.mle_retvals.get("converged", True),
        })
        done += 1
        if done % 5 == 0:
            print(f"  [{done}/{total}] SARIMAX{(p,d,q)}x{(P,D,Q,m)}  "
                  f"AIC={fit.aic:.1f}  BIC={fit.bic:.1f}  "
                  f"{time.time()-t0:.0f}s")
    except Exception as e:
        print(f"  FAIL  SARIMAX{(p,d,q)}x{(P,D,Q,m)}: {e}")

grid_df = pd.DataFrame(results).sort_values("aic").reset_index(drop=True)
print("\nTop 10 by AIC:")
print(grid_df.head(10).to_string(index=False))


In [ ]:
# Save grid search results
import json
(PROJECT_ROOT / "results").mkdir(exist_ok=True)
grid_df_serializable = grid_df.copy()
grid_df_serializable["order"]    = grid_df_serializable["order"].astype(str)
grid_df_serializable["seasonal"] = grid_df_serializable["seasonal"].astype(str)
grid_df_serializable.to_csv(
    PROJECT_ROOT / "results" / f"sarima_grid_search_{SITE}.csv", index=False
)

# Heatmap: AIC by (p,q) for best (P,Q)
best_PQ = grid_df.iloc[0][["seasonal"]].values[0]
P_best, Q_best = grid_df.iloc[0]["seasonal"][:2]

sub = grid_df[grid_df["seasonal"].apply(lambda s: s[0] == P_best and s[1] == Q_best)].copy()
sub["p"] = sub["order"].apply(lambda x: x[0])
sub["q"] = sub["order"].apply(lambda x: x[2])

pivot = sub.pivot(index="p", columns="q", values="aic")
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot, aspect="auto", cmap="YlGnBu_r")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
ax.set_xlabel("q"); ax.set_ylabel("p")
ax.set_title(f"AIC heatmap — SARIMAX(p,1,q)×({P_best},1,{Q_best},24) — {SITE}")
plt.colorbar(im, ax=ax, label="AIC")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.iloc[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / f"sarima_aic_heatmap_{SITE}.png", dpi=120)
plt.show()

## 5. Residual diagnostics on the selected model

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
import scipy.stats as stats

# Fit the order selected from the grid-search/ACF-PACF analysis above on the
# full training set. Update CHOSEN_ORDER / CHOSEN_SEASONAL to match the
# best model found for raw GHI (do not assume the clear-sky-index values
# (2,1,2)(1,1,1)[24] are still optimal — re-derive them from this series).
CHOSEN_ORDER    = (2, 1, 2)
CHOSEN_SEASONAL = (1, 1, 1, 24)

print(f"Fitting SARIMAX{CHOSEN_ORDER}x{CHOSEN_SEASONAL} on full training set ({len(ghi_train):,} obs)...")
t0 = time.time()
mod_final = SARIMAX(
    ghi_train,
    order=CHOSEN_ORDER,
    seasonal_order=CHOSEN_SEASONAL,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
fit_final = mod_final.fit(disp=False, maxiter=200)
print(f"Done in {time.time()-t0:.0f}s")
print(fit_final.summary())


In [ ]:
resid = fit_final.resid.dropna()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Residual time series
axes[0, 0].plot(resid.index[-500:], resid.values[-500:], lw=0.6)
axes[0, 0].axhline(0, color="r", lw=0.8, ls="--")
axes[0, 0].set_title("Residuals (last 500 obs)")
axes[0, 0].set_ylabel("residual")

# Histogram + normal fit
axes[0, 1].hist(resid, bins=60, density=True, alpha=0.7)
xr = np.linspace(resid.min(), resid.max(), 200)
axes[0, 1].plot(xr, stats.norm.pdf(xr, resid.mean(), resid.std()), "r-", lw=1.5)
axes[0, 1].set_title("Residual distribution")

# Q-Q plot
stats.probplot(resid, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("Q-Q plot")

# ACF of residuals
plot_acf(resid, lags=72, ax=axes[1, 1], title="ACF of residuals (lags 0–72)")
axes[1, 1].axvline(24, color="r", ls="--", lw=0.8, label="lag=24")
axes[1, 1].axvline(48, color="r", ls="--", lw=0.8)
axes[1, 1].legend(fontsize=8)

fig.suptitle(f"Residual diagnostics — SARIMAX{CHOSEN_ORDER}×{CHOSEN_SEASONAL} — {SITE}", fontsize=12)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / f"sarima_residuals_{SITE}.png", dpi=120)
plt.show()

In [ ]:
# Ljung-Box test for residual autocorrelation
lb = acorr_ljungbox(resid, lags=[12, 24, 36, 48], return_df=True)
print("Ljung-Box test on residuals:")
print(lb)
print()
print("H0: residuals are uncorrelated (white noise).")
print("p > 0.05 → fail to reject H0 → residuals are approximately white noise (good).")

## 6. Comparison: chosen order vs alternatives

In [ ]:
# Show AIC/BIC for a few candidate models for comparison
# (fitted on the sub-sample used in the grid search)
candidates = [
    {"order": (1, 1, 1), "seasonal": (1, 1, 1, 24), "label": "ARIMA(1,1,1)(1,1,1)[24]"},
    {"order": (2, 1, 2), "seasonal": (1, 1, 1, 24), "label": "ARIMA(2,1,2)(1,1,1)[24]"},
    {"order": (3, 1, 2), "seasonal": (1, 1, 1, 24), "label": "ARIMA(3,1,2)(1,1,1)[24]"},
    {"order": (2, 1, 3), "seasonal": (1, 1, 1, 24), "label": "ARIMA(2,1,3)(1,1,1)[24]"},
    {"order": (2, 1, 2), "seasonal": (0, 1, 1, 24), "label": "ARIMA(2,1,2)(0,1,1)[24]"},
    {"order": (2, 1, 2), "seasonal": (1, 1, 0, 24), "label": "ARIMA(2,1,2)(1,1,0)[24]"},
]

comp_rows = []
for c in candidates:
    t0 = time.time()
    try:
        m = SARIMAX(
            ghi_search,
            order=c["order"],
            seasonal_order=c["seasonal"],
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False, maxiter=150)
        comp_rows.append({"Model": c["label"], "AIC": m.aic, "BIC": m.bic,
                           "Params": m.df_model, "Time(s)": f"{time.time()-t0:.0f}"})
    except Exception as e:
        comp_rows.append({"Model": c["label"], "AIC": float("nan"), "BIC": float("nan"),
                           "Params": None, "Time(s)": "FAIL"})
        print(f"  FAIL {c['label']}: {e}")

comp_df = pd.DataFrame(comp_rows).sort_values("AIC").reset_index(drop=True)
comp_df["ΔAIC"] = comp_df["AIC"] - comp_df["AIC"].min()
comp_df["ΔBIC"] = comp_df["BIC"] - comp_df["BIC"].min()
print(comp_df.to_string(index=False))
comp_df.to_csv(PROJECT_ROOT / "results" / f"sarima_model_comparison_{SITE}.csv", index=False)


## 7. Summary and order decision

Fill in after running cells above.

In [ ]:
print("="*60)
print(f"SARIMA Order Selection Summary — site: {SITE}")
print("="*60)

print(f"""
Data:
  Series: raw hourly GHI (W/m²) — same tabular ground-truth as DL models,
          no clear-sky normalisation or derived index
  Resolution: hourly (m=24, daily seasonality)
  Training obs: {len(ghi_train):,}

Integration orders (from stationarity tests — fill in after running cells above):
  d=<fill in>   — regular differencing needed to make GHI stationary
  D=<fill in>   — seasonal differencing needed to remove the diurnal cycle

AR/MA orders (from ACF/PACF of Δ Δ₂₄ GHI — fill in after inspection):
  Non-seasonal: p=<fill in>  q=<fill in>
  Seasonal:     P=<fill in>  Q=<fill in>

AIC/BIC grid search:
  Best model by AIC: see grid_df / comp_df above
  <fill in chosen order and ΔAIC vs. best>

Residual diagnostics (chosen model on full training set):
  Ljung-Box: <fill in p-values from cell above>
  ACF residuals: <describe whether remaining autocorrelation is present>

Decision:
  SARIMAX(<p>,<d>,<q>)(<P>,<D>,<Q>)[24]   — UPDATE SARIMA_ORDER /
  SARIMA_SEASONAL_ORDER in 05_sarima_baseline.py to match.
  Justified by: stationarity tests, ACF/PACF inspection, AIC/BIC comparison,
  and acceptable Ljung-Box p-values on residuals, all computed directly on
  the raw hourly GHI series (the same physical quantity and tabular source
  used to train and evaluate persistence and the deep-learning models).
""")
